# Experiment — YOLO11s Instance Segmentation with Single Object Class

## Objective

This experiment simplifies the inventory instance-segmentation task by combining all 109 inventory categories into a single class:

**Class 0 — object**

The original images, segmentation polygons, and train/validation/test splits are preserved.

Only the class ID in each YOLO segmentation annotation is changed to `0`.

A larger YOLO11 model is also used:

- Previous model: YOLO11n-Seg
- New model: YOLO11s-Seg
- Number of classes: 1
- Class name: `object`
- Image size: 640 × 640
- Maximum epochs: 100
- Batch size: 8
- Patience: 20

The purpose is to evaluate whether simplifying the classification task and increasing model capacity improves object detection and segmentation performance.

## 1. Setup and Project Paths

This section defines the original 109-class dataset and the new single-class dataset.

The original dataset will remain unchanged.

A separate dataset will be created for the single-class experiment.

In [1]:
# ============================================================
# CELL 1 — SETUP & PATHS
# ============================================================

from pathlib import Path
import shutil
import yaml
import torch
import ultralytics

project_path = Path(r"G:\AIIC")

segmentation_project = (
    project_path
    / "yolo_segmentation"
)

# Original 109-class dataset
original_dataset = (
    segmentation_project
    / "dataset"
)

# New single-class dataset
single_class_dataset = (
    segmentation_project
    / "dataset_single_class"
)

runs_path = (
    segmentation_project
    / "runs"
)

single_class_yaml = (
    single_class_dataset
    / "data.yaml"
)

print("=" * 70)
print("SINGLE-CLASS YOLO11s EXPERIMENT")
print("=" * 70)

print("Ultralytics :", ultralytics.__version__)
print("PyTorch     :", torch.__version__)
print("CUDA        :", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU         :",
        torch.cuda.get_device_name(0)
    )

print()
print("Original dataset :", original_dataset)
print("New dataset      :", single_class_dataset)
print("New YAML         :", single_class_yaml)

print()
print(
    "Original dataset exists:",
    original_dataset.exists()
)

print("=" * 70)

SINGLE-CLASS YOLO11s EXPERIMENT
Ultralytics : 8.4.137
PyTorch     : 2.13.0+cu130
CUDA        : True
GPU         : NVIDIA GeForce RTX 5060

Original dataset : G:\AIIC\yolo_segmentation\dataset
New dataset      : G:\AIIC\yolo_segmentation\dataset_single_class
New YAML         : G:\AIIC\yolo_segmentation\dataset_single_class\data.yaml

Original dataset exists: True


## 2. Create the Single-Class Dataset

A new dataset folder is created so that the original 109-class dataset remains untouched.

The train, validation, and test images are copied without modification.

The annotation files will be recreated separately with all class IDs converted to `0`.

In [2]:
# ============================================================
# CELL 2 — CREATE SINGLE-CLASS DATASET
# ============================================================

if single_class_dataset.exists():
    raise FileExistsError(
        f"{single_class_dataset} already exists.\n"
        "Delete it manually only if you intentionally want "
        "to rebuild the single-class dataset."
    )

# Create dataset root
single_class_dataset.mkdir(
    parents=True,
    exist_ok=False
)

# ------------------------------------------------------------
# COPY IMAGES
# ------------------------------------------------------------

for split in [
    "train",
    "val",
    "test"
]:

    source_images = (
        original_dataset
        / "images"
        / split
    )

    destination_images = (
        single_class_dataset
        / "images"
        / split
    )

    shutil.copytree(
        source_images,
        destination_images
    )

    # Create empty labels directory
    (
        single_class_dataset
        / "labels"
        / split
    ).mkdir(
        parents=True,
        exist_ok=True
    )

    image_count = len(
        [
            p
            for p in destination_images.iterdir()
            if p.suffix.lower()
            in [
                ".jpg",
                ".jpeg",
                ".png"
            ]
        ]
    )

    print(
        f"{split:5s} images copied:",
        image_count
    )

print()
print(
    "✅ Single-class dataset folders created."
)

train images copied: 1909
val   images copied: 403
test  images copied: 414

✅ Single-class dataset folders created.


## 3. Convert All Segmentation Labels to One Class

Each annotation line in the original dataset begins with a class ID followed by segmentation polygon coordinates.

For this experiment, only the class ID is modified.

All original class IDs from `0–108` are replaced with:

`0 = object`

The segmentation polygon coordinates are preserved exactly.

In [3]:
# ============================================================
# CELL 3 — CONVERT ALL CLASS IDs TO 0
# ============================================================

conversion_summary = {}

for split in [
    "train",
    "val",
    "test"
]:

    source_labels = (
        original_dataset
        / "labels"
        / split
    )

    destination_labels = (
        single_class_dataset
        / "labels"
        / split
    )

    label_files = list(
        source_labels.glob("*.txt")
    )

    total_instances = 0
    converted_instances = 0

    for source_file in label_files:

        destination_file = (
            destination_labels
            / source_file.name
        )

        new_lines = []

        with open(
            source_file,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                parts = line.split()

                total_instances += 1

                # Replace ONLY class ID
                parts[0] = "0"

                new_lines.append(
                    " ".join(parts)
                )

                converted_instances += 1

        with open(
            destination_file,
            "w",
            encoding="utf-8"
        ) as f:

            if new_lines:

                f.write(
                    "\n".join(new_lines)
                    + "\n"
                )

    conversion_summary[split] = {
        "labels": len(label_files),
        "instances": converted_instances
    }

    print(
        f"{split:5s} | "
        f"label files: {len(label_files):4d} | "
        f"instances converted: "
        f"{converted_instances:5d}"
    )

print()
print(
    "✅ All annotation class IDs converted to 0."
)

train | label files: 1909 | instances converted:  4355
val   | label files:  403 | instances converted:   811
test  | label files:  414 | instances converted:   901

✅ All annotation class IDs converted to 0.


## 4. Create Single-Class Dataset Configuration

The new YOLO dataset configuration contains only one class.

The class is named:

`object`

Therefore:

- `nc = 1`
- Class ID `0 = object`

In [4]:
# ============================================================
# CELL 4 — CREATE SINGLE-CLASS DATA.YAML
# ============================================================

single_class_config = {
    "path": single_class_dataset.as_posix(),

    "train": "images/train",
    "val": "images/val",
    "test": "images/test",

    "nc": 1,

    "names": {
        0: "object"
    }
}

with open(
    single_class_yaml,
    "w",
    encoding="utf-8"
) as f:

    yaml.safe_dump(
        single_class_config,
        f,
        sort_keys=False,
        allow_unicode=True
    )

print("=" * 70)
print("SINGLE-CLASS DATA.YAML")
print("=" * 70)

with open(
    single_class_yaml,
    "r",
    encoding="utf-8"
) as f:

    print(f.read())

print("=" * 70)

SINGLE-CLASS DATA.YAML
path: G:/AIIC/yolo_segmentation/dataset_single_class
train: images/train
val: images/val
test: images/test
nc: 1
names:
  0: object



## 5. Verify the Single-Class Dataset

Before training, all generated annotation files are checked.

Every segmentation annotation must use class ID `0`.

The number of images and segmentation instances is also reported for each dataset split.

In [5]:
# ============================================================
# CELL 5 — VERIFY SINGLE-CLASS LABELS
# ============================================================

all_valid = True

print("=" * 70)
print("SINGLE-CLASS DATASET VERIFICATION")
print("=" * 70)

for split in [
    "train",
    "val",
    "test"
]:

    labels_path = (
        single_class_dataset
        / "labels"
        / split
    )

    images_path = (
        single_class_dataset
        / "images"
        / split
    )

    class_ids_found = set()
    total_instances = 0

    for label_file in labels_path.glob("*.txt"):

        with open(
            label_file,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = line.strip()

                if not line:
                    continue

                class_id = int(
                    float(
                        line.split()[0]
                    )
                )

                class_ids_found.add(
                    class_id
                )

                total_instances += 1

    image_count = len(
        [
            p
            for p in images_path.iterdir()
            if p.suffix.lower()
            in [
                ".jpg",
                ".jpeg",
                ".png"
            ]
        ]
    )

    valid_split = (
        class_ids_found == {0}
    )

    if not valid_split:
        all_valid = False

    print(
        f"{split.upper():5s}"
    )

    print(
        "  Images       :",
        image_count
    )

    print(
        "  Instances    :",
        total_instances
    )

    print(
        "  Class IDs    :",
        sorted(class_ids_found)
    )

    print(
        "  Valid        :",
        valid_split
    )

    print()


print("=" * 70)

if all_valid:

    print(
        "✅ VERIFIED: ALL DATASET LABELS USE CLASS 0 ONLY"
    )

else:

    print(
        "❌ VERIFICATION FAILED — DO NOT TRAIN"
    )

print("=" * 70)

SINGLE-CLASS DATASET VERIFICATION
TRAIN
  Images       : 1909
  Instances    : 4355
  Class IDs    : [0]
  Valid        : True

VAL  
  Images       : 403
  Instances    : 811
  Class IDs    : [0]
  Valid        : True

TEST 
  Images       : 414
  Instances    : 901
  Class IDs    : [0]
  Valid        : True

✅ VERIFIED: ALL DATASET LABELS USE CLASS 0 ONLY


## 6. YOLO11s-Seg Single-Class Smoke Test

Before full training, a one-epoch smoke test is performed using YOLO11s-Seg.

This experiment uses:

- Model: YOLO11s-Seg
- Number of classes: 1
- Class: `object`
- Dataset: Single-class inventory dataset
- Image size: 640 × 640
- Batch size: 8
- Epochs: 1

The purpose of the smoke test is to confirm that the new single-class dataset and YOLO11s-Seg training pipeline work correctly before running the full 100-epoch experiment.

In [6]:
# ============================================================
# CELL 6 — YOLO11s SINGLE-CLASS SMOKE TEST
# 1 EPOCH ONLY
# ============================================================

from ultralytics import YOLO

print("=" * 70)
print("YOLO11s-SEG SINGLE-CLASS SMOKE TEST")
print("=" * 70)

print("Model        : YOLO11s-Seg")
print("Classes      : 1")
print("Class Name   : object")
print("Epochs       : 1")
print("Batch Size   : 8")
print("Image Size   : 640")
print("Dataset YAML :", single_class_yaml)

print("=" * 70)

# ------------------------------------------------------------
# LOAD FRESH PRETRAINED YOLO11s-SEG
# ------------------------------------------------------------

smoke_model = YOLO(
    "yolo11s-seg.pt"
)

# ------------------------------------------------------------
# 1-EPOCH SMOKE TEST
# ------------------------------------------------------------

smoke_results = smoke_model.train(
    data=str(single_class_yaml),

    epochs=1,
    imgsz=640,
    batch=8,

    device=0,
    workers=4,

    patience=20,

    pretrained=True,
    optimizer="auto",

    amp=True,
    cache=False,

    project=str(runs_path),

    name="yolo11s_single_class_smoke_test",

    exist_ok=True,

    plots=True,
    verbose=True
)

print()
print("=" * 70)
print("✅ YOLO11s SINGLE-CLASS SMOKE TEST COMPLETED")
print("=" * 70)

YOLO11s-SEG SINGLE-CLASS SMOKE TEST
Model        : YOLO11s-Seg
Classes      : 1
Class Name   : object
Epochs       : 1
Batch Size   : 8
Image Size   : 640
Dataset YAML : G:\AIIC\yolo_segmentation\dataset_single_class\data.yaml
New https://pypi.org/project/ultralytics/8.4.147 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.137  Python-3.14.3 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060, 8123MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\AIIC\yolo_segmentation\dataset_single_class\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, 

## 7. Full Training — YOLO11s-Seg Single-Class Model

After successfully completing the one-epoch smoke test, the YOLO11s-Seg model is trained on the full single-class inventory dataset.

All 109 original inventory categories are represented as one class:

`0 = object`

The model focuses only on detecting and segmenting individual physical objects.

### Training Configuration

- Model: YOLO11s-Seg
- Initial weights: `yolo11s-seg.pt`
- Number of classes: 1
- Class name: `object`
- Image size: 640 × 640
- Maximum epochs: 100
- Batch size: 8
- Early stopping patience: 20
- Optimizer: Auto
- Device: CUDA GPU
- Loss: Default YOLO11 segmentation loss

The model is initialized again from the original pretrained YOLO11s-Seg weights rather than continuing from the smoke-test model.

In [ ]:
# ============================================================
# CELL 7 — FULL YOLO11s SINGLE-CLASS TRAINING
# ============================================================

from ultralytics import YOLO

print("=" * 70)
print("FULL YOLO11s-SEG SINGLE-CLASS TRAINING")
print("=" * 70)

print("Model        : YOLO11s-Seg")
print("Dataset      : Single-Class Inventory Dataset")
print("Classes      : 1")
print("Class Name   : object")
print("Epochs       : 100")
print("Batch Size   : 8")
print("Image Size   : 640")
print("Patience     : 20")

print("=" * 70)


# ------------------------------------------------------------
# FRESH PRETRAINED MODEL
# ------------------------------------------------------------

single_class_model = YOLO(
    "yolo11s-seg.pt"
)


# ------------------------------------------------------------
# FULL TRAINING
# ------------------------------------------------------------

single_class_results = single_class_model.train(

    data=str(single_class_yaml),

    epochs=100,
    imgsz=640,
    batch=8,

    device=0,
    workers=4,

    patience=20,

    pretrained=True,
    optimizer="auto",

    amp=True,
    cache=False,

    project=str(runs_path),

    name="yolo11s_seg_aiic_single_class",

    exist_ok=False,

    plots=True,
    verbose=True
)


print()
print("=" * 70)
print("✅ FULL SINGLE-CLASS TRAINING COMPLETED")
print("=" * 70)

FULL YOLO11s-SEG SINGLE-CLASS TRAINING
Model        : YOLO11s-Seg
Dataset      : Single-Class Inventory Dataset
Classes      : 1
Class Name   : object
Epochs       : 100
Batch Size   : 8
Image Size   : 640
Patience     : 20
New https://pypi.org/project/ultralytics/8.4.147 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.137  Python-3.14.3 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060, 8123MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\AIIC\yolo_segmentation\dataset_single_class\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, e